### 1. Load a paper
Use our own `fetch_paper()` from `src/paper_companion/ingest/`. The HTML path gives 8 clean sections.

In [2]:
from paper_companion.ingest.arxiv import fetch_paper
paper = fetch_paper("1706.03762")
print(paper["source"], len(paper["sections"]), "sections")

html 8 sections


### 2. Chunk, embed, search
Cut the paper into small overlapping chunks, turn them into vectors, and find the ones closest to a question.

In [4]:
def chunk_words(text, size=200, overlap=40):
    words = text.split()
    step = size - overlap
    return [" ".join(words[i:i + size]) for i in range(0, max(len(words) - overlap, 1), step)]

chunks = []
for s in paper["sections"]:
    for i, c in enumerate(chunk_words(s["text"])):
        chunks.append({"section": s["title"], "chunk": i, "text": c})

print(len(chunks), "chunks")
for c in chunks[:6]:
    print(f"{c['section'][:30]:30} #{c['chunk']}  {len(c['text'].split())} words")

29 chunks
Abstract                       #0  169 words
1 Introduction                 #0  200 words
1 Introduction                 #1  145 words
2 Background                   #0  200 words
2 Background                   #1  125 words
3 Model Architecture           #0  200 words


#### See the overlap
The last 40 words of one chunk are the first 40 of the next, so a sentence cut at a border still appears whole somewhere.

In [5]:
a, b = chunks[3]["text"].split(), chunks[4]["text"].split()
print("END of chunk 3:  ", " ".join(a[-40:]))
print("START of chunk 4:", " ".join(b[:40]))

END of chunk 3:   has been used successfully in a variety of tasks including reading comprehension, abstractive summarization, textual entailment and learning task-independent sentence representations [ 4 , 27 , 28 , 22 ] . End-to-end memory networks are based on a recurrent attention
START of chunk 4: has been used successfully in a variety of tasks including reading comprehension, abstractive summarization, textual entailment and learning task-independent sentence representations [ 4 , 27 , 28 , 22 ] . End-to-end memory networks are based on a recurrent attention


### 3. Embed the chunks
Turn each chunk into a vector of numbers that captures its meaning, using bge-small

In [6]:
from sentence_transformers import SentenceTransformer
emb_model = SentenceTransformer("BAAI/bge-small-en-v1.5")
vectors = emb_model.encode([c["text"] for c in chunks], normalize_embeddings=True)
print(vectors.shape)
print(vectors[0][:8])


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1789.33it/s]


(29, 384)
[-0.0599346  -0.02490155 -0.04616825 -0.01890551  0.0020497  -0.00197272
 -0.03951125  0.02201432]


### 4. Search with a question
Embed the question, score every chunk by cosine similarity (dot product of normalized vectors), show the top 3.


In [7]:
query = "What are end-to-end memory networks based on?"
q = emb_model.encode(query, normalize_embeddings=True)
scores = vectors @ q
for i in scores.argsort()[::-1][:3]:
    print(f"{scores[i]:.3f}  {chunks[i]['section']} #{chunks[i]['chunk']}  {chunks[i]['text'][:120]}")

0.728  4 Why Self-Attention #0  In this section we compare various aspects of self-attention layers to the recurrent and convolutional layers commonly u
0.726  3 Model Architecture #0  Most competitive neural sequence transduction models have an encoder-decoder structure [ 5 , 2 , 35 ] . Here, the encode
0.718  2 Background #1  has been used successfully in a variety of tasks including reading comprehension, abstractive summarization, textual ent


### 5. Query instruction prefix
bge expects search questions to start with an instruction. Does it move the right chunk to #1?

In [8]:
prefix = "Represent this sentence for searching relevant passages: "
q2 = emb_model.encode(prefix + query, normalize_embeddings=True)
scores2 = vectors @ q2
for i in scores2.argsort()[::-1][:3]:
    print(f"{scores2[i]:.3f}  {chunks[i]['section']} #{chunks[i]['chunk']}  {chunks[i]['text'][:120]}")

0.704  2 Background #1  has been used successfully in a variety of tasks including reading comprehension, abstractive summarization, textual ent
0.695  3 Model Architecture #0  Most competitive neural sequence transduction models have an encoder-decoder structure [ 5 , 2 , 35 ] . Here, the encode
0.679  2 Background #0  The goal of reducing sequential computation also forms the foundation of the Extended Neural GPU [ 16 ] , ByteNet [ 18 ]


### 6. The pipeline as a package
fetch → chunk → embed → save → search, all from `src/paper_companion/`.

In [1]:
from paper_companion.ingest.arxiv import fetch_paper
from paper_companion.index.vectors import build_index, save_index
from paper_companion.retrieve.search import search

paper = fetch_paper("1706.03762")
chunks, vectors = build_index(paper)
save_index(paper["arxiv_id"], chunks, vectors)
for r in search("How many layers does the encoder have?", chunks, vectors):
    print(f"{r['score']:.3f}  {r['section']} #{r['chunk']}  {r['text'][:100]}")

/Users/ibrahim/paper-companion/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1112 11:32:57.354000 80929 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5499.80it/s]


0.748  3 Model Architecture #0  Most competitive neural sequence transduction models have an encoder-decoder structure [ 5 , 2 , 35 
0.744  3 Model Architecture #5  d_{k}}$ , $W^{K}_{i}\in\mathbb{R}^{d_{\text{model}}\times d_{k}}$ , $W^{V}_{i}\in\mathbb{R}^{d_{\tex
0.744  3 Model Architecture #6  layer in the encoder. Each position in the encoder can attend to all positions in the previous layer
